# VGG16 Grid Search

In [1]:
pip install keras-tuner

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16
from keras_tuner import RandomSearch
from tensorflow.keras.preprocessing import image_dataset_from_directory

# Define directories for your datasets
train_data_dir = '/Users/et/code/Lucia-Cordero/ReefSight-Project/raw_data/image_data/train_validation_test/train'
val_data_dir = '/Users/et/code/Lucia-Cordero/ReefSight-Project/raw_data/image_data/train_validation_test/val'
test_data_dir = '/Users/et/code/Lucia-Cordero/ReefSight-Project/raw_data/image_data/train_validation_test/test'

# Set parameters
batch_size = 16
seed = 42
image_size = (224, 224)
num_classes = 2

# Load datasets
train_ds = image_dataset_from_directory(
    train_data_dir,
    labels="inferred",
    label_mode="binary",
    seed=seed,
    image_size=image_size,
    batch_size=batch_size
)

val_ds = image_dataset_from_directory(
    val_data_dir,
    labels="inferred",
    label_mode="binary",
    seed=seed,
    image_size=image_size,
    batch_size=batch_size
)

test_ds = image_dataset_from_directory(
    test_data_dir,
    labels="inferred",
    label_mode="binary",
    seed=seed,
    image_size=image_size,
    batch_size=batch_size
)

# Data Augmentation
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal_and_vertical", input_shape=image_size + (3,)),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
])

# Load the VGG16 model without the top layer
base_model = VGG16(weights='imagenet', include_top=False, input_shape=image_size + (3,))

# Define the build model function for Keras Tuner
def build_model(hp):
    model = models.Sequential()

    # Add a Rescaling layer
    model.add(layers.Rescaling(1./255, input_shape=image_size + (3,)))  # Rescale pixel values to [0, 1]

    # Add data augmentation
    model.add(data_augmentation)

    # Add VGG16 base model
    model.add(base_model)

    # Fine-tuning decision
    unfreeze_layers = hp.Boolean('unfreeze_layers')  # New hyperparameter

    # Unfreeze the last few layers if specified
    if unfreeze_layers:
        for layer in base_model.layers[-4:]:  # Unfreeze the last 4 layers
            layer.trainable = True
    else:
        for layer in base_model.layers:
            layer.trainable = False

    # Add additional layers
    model.add(layers.GlobalAveragePooling2D())
    model.add(layers.Dense(64, activation='relu'))
    model.add(layers.Dropout(hp.Float('dropout_rate', 0.2, 0.5, step=0.1)))
    model.add(layers.Dense(1, activation='sigmoid'))  # For binary classification

    # Compile the model
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=hp.Choice('learning_rate', [0.0001, 0.001, 0.01])),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

# Set up Keras Tuner
tuner = RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=24,  # Adjust as needed
    executions_per_trial=1,
    directory='hyperparameter_tuning',
    project_name='vgg16_cnn_tuning'
)

# Run the hyperparameter search
tuner.search(train_ds,
             validation_data=val_ds,
             epochs=1000,
             callbacks=[
                 tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=10),
                 tf.keras.callbacks.ModelCheckpoint(filepath='best_vgg16_model.keras', monitor='val_accuracy', save_best_only=True),
                 tf.keras.callbacks.ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=4, verbose=1)
             ])

# Retrieve and summarize the best model
best_model = tuner.get_best_models(num_models=1)[0]
best_hyperparameters = tuner.get_best_hyperparameters(num_trials=1)[0]

print("Best Hyperparameters:", best_hyperparameters.values)


Trial 17 Complete [00h 04m 13s]
val_accuracy: 0.9239130616188049

Best val_accuracy So Far: 0.945652186870575
Total elapsed time: 02h 18m 52s
Best Hyperparameters: {'unfreeze_layers': True, 'dropout_rate': 0.2, 'learning_rate': 0.001}


/Users/et/.pyenv/versions/3.10.6/envs/ReefSight-Project/lib/python3.10/site-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 62 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
